#### inputs
##### constants, sample parameters, microscope parameters

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from tifffile import imread
from matplotlib import colormaps
import os
from pathlib import Path
import pandas as pd
from typing import List
from dataclasses import replace
import utils_bone
import get_images_and_metadata as iam
import resolution_theory as thc
import sweep_func as sp
import mean_free_path_func as mfp
import utils_microscope
from utils_microscope import STEMInputs

In [2]:
from dataclasses import dataclass

@dataclass
class STEMInputs:
    # Sample params
    matrix: str = "water"
    obj: str = "bone"
    t: float = 1.0e-6           # matrix thickness [m]
    t_SiN: float = 50e-9        # SiN window thickness [m]
    z_rel: float = 1.0          # relative position (0..1) from TOP window (STEM SI)
    x: float = 2.0e-6           # lamella thickness

    # Microscope settings
    U_eV: float = 200_000.0     # accelerating voltage [eV]
    Cs_m: float = 1e-3          # spherical aberration [m]
    alpha_p: float = 1.7e-3       # probe semi-convergence [rad]
    beta_rad: float = 0.03      # detector opening semi-angle β [rad] (set 0.002–0.03 for BF)
    eD_m2: float = 100 * 1e20   # dose eD [e-/m^2]; 100 e-/Å^2 -> 100*1e20

    # Detector split
    beta_BF_DF_threshold: float = 0.030  # 30 mrad

    # Fundamental constants (use same symbols as SI)
    epsi0: float = 8.854188e-12
    h: float = 6.626e-34
    m0: float = 9.109e-31
    c: float = 2.998e8
    e: float = 1.602e-19
    NA: float = 6.022e23
    aH: float = 5.292e-11
    SNRf: float = 3.0

    # Material properties

    # defaults for gold
    rho_o: float = 19.3e6
    Z_o: float = 79.0
    W_o: float = 197.0

    # defaults for bone
    rho_b: float = 2e6  #g/m^3
    Z_b: float = 11.72
    W_b: float = 23.6  #g/mol

    # defaults for matrix (water)
    rho_m: float = 997000.0
    Z_m: float = 4.7
    W_m: float = 6.0

    # defaults for SiN
    rho_SiN: float = 3.2e6
    Z_SiN: float = 10.6
    W_SiN: float = 20.0

In [3]:
inp = STEMInputs

#### dose & therotical resoltion functions

In [4]:
def depth_of_focus(alpha_mrad):
    DOFs = []
    EE = inp.e * inp.U_eV
    E0 = inp.m0 * (inp.c**2)
    lam = inp.h * inp.c / np.sqrt(2* EE * E0 + EE ** 2)
    dof_nm = 2 * 1e15 * lam / alpha_mrad**2
    return dof_nm

In [5]:
def total_electron_dose(current, dwell, pixel, sparsity):
    """
    Calculate electron dose per image.
    Dose = (current * dwell) / (pixel^2)
    current in pA
    dwell in us
    pixel in nm
    dose in e/A2
    """

    dose = (0.0625 * current * dwell / (pixel ** 2)) / (100 / sparsity)

    return dose

In [6]:
import numpy as np

def theoretical_resolution(inp: STEMInputs, alpha_mrad):
    """
    alpha in mrad
    theoretical resolution in nm
    """
    theor_res = []
    EE = inp.e * inp.U_eV
    E0 = inp.m0 * (inp.c ** 2)
    lam = inp.h * inp.c / np.sqrt(2 * EE * E0 + EE ** 2)
    theor_res = 1e9 * lam / (2 * (alpha_mrad * 1e-3)) 
    return theor_res

In [7]:
inp = STEMInputs()
def possible_res(inp: STEMInputs, alpha, dose): 
    """
    inputs:
    convergence angle [mrad]
    dose [e/A2]
    """
    inpi = replace(inp, eD_m2=dose * 1e20)
    inpii = replace(inpi, alpha_p=alpha * 1e-3)
    out = thc.compute_stem_resolution_for_beta(inpii) 
    return out["dSTEM_nm"]

#### calls for theoretical values

In [8]:
#For dose calculation 
### input: current [pA], dwell [us], pixel [nm], sparsity [%]
### output: dose [e/A2]

dose = total_electron_dose(18, 2, 1.212, 20) 
print(f"dose = {dose:.3f} e-/A2")

dose = 0.306 e-/A2


In [9]:
#For theoretical resolution - wavelength divided by (2 * alpha)
### input: convergence angle [mrad]
### output: theoretical resolution [nm]

theor_res = theoretical_resolution(inp, 1.7)
print(f"theoretical resolution = {theor_res:.3f} nm")

theoretical resolution = 0.738 nm


In [10]:
#Possible resolution taking into account snr, blur, probe, and material - based on de Jonge 2018
### input: convergence angle [mrad], dose [e/A2]
### replaces dose and alpha in STEMInputs and uses the function: thc.compute_stem_resolution_for_beta(STEMInputs) 
### output: possible /limited resolution [nm]

poss_res = possible_res(inp, 1.7, 1.532)
print(f"possible resolution = {poss_res:.3f} nm")

possible resolution = 1.668 nm


In [11]:
#camera length [mm]
CL = 7.0 / math.tan(inp.beta_rad)
print(f"camera length = {CL:.0f} mm")

camera length = 233 mm


In [12]:
from mean_free_path_func import mean_free_path
from mean_free_path_func import compute_effective_mfp

inp2 = replace(inp, x = 200 * 1e-9, t = 0 * 1e-9)
l_b, l_m, l_SiN, l_o = mean_free_path(inp)

eff_mfp = compute_effective_mfp(inp2.x, inp2.t, inp2.t_SiN, l_b, l_m, l_SiN)
print(f"mfp of bone = {l_b * 1e9:.0f} nm")
print(f"mfp of SiN windows = {2* l_SiN * 1e9:.0f} nm")
print(f"effective mfp of chip = {eff_mfp * 1e9:.0f} nm")

mfp of bone = 695 nm
mfp of SiN windows = 878 nm
effective mfp of chip = 610 nm
